# 1. Introduction

1. 왜 ‘실시간’ 영상 생성인가?

기존 영상 생성 모델은 다음과 같이 작동
프롬프트 입력 → 수분간 대기 → 완성된 영상 전체 출력

이 방식은 영화나 광고처럼 미리 영상을 만드는 작업에는 적합하지만, 대화·게임·라이브 방송처럼 사용자가 즉시 개입해야 하는 상황에는 적합하지 않음

저자들은 실시간 콘텐츠는 사용자마다 별도의 생성이 필요하지만, 오프라인 영상은 한 번 만든 뒤 여러 사람이 볼 수 있으므로 장기적으로는 실시간 생성 수요가 훨씬 클 것이라고 주장

다만 이 수요 계산은 엄밀한 시장 분석이라기보다 연구 방향의 중요성을 강조하기 위한 직관적 논증에 가까움.

rom a demand perspective, suppose that each user has an
average demand of α∈[0,1] for real-time interactive visual content, e.g., α= 0.5. Then the total demand
scales with α×N, where N is the number of users. In contrast, suppose that each user has an average demand
of β ∈[0,1] for offline-generated visual content, e.g., β = 0.5. Since offline-generated videos can be replayed
and shared, their generation demand scales more like β×N/m, where mis the average number of views per
generated video. If we assume α≈βand m>100, then the demand for real-time interactive video generation
is much greater than that for offline-generated videos.

2. About Vidu S1

S1은 다음 세 가지를 처음 구축한 기반 모델입니다.
1. 음성으로 미래 행동을 실시간 제어
2. 자기회귀 방식으로 무한 길이 영상 생성
3. 몇 단계만으로 빠르게 생성하여 실시간 속도 달성

S2도 이 실시간 지시 구조를 그대로 사용. S2의 차이는 춤과 같은 더 큰 신체 동작까지 지시 이행 범위를 확장했다는 것.

In [ ]:
Sliding-window streaming inference

생성된 모든 과거 프레임을 계속 attention에 넣으면 영상 길이에 따라 메모리와 연산량이 증가합니다.
S1은 전체 과거가 아니라 다음 세 부분만 봅니다.
1. 최초 참조 이미지와 첫 생성 구간
2. 고정된 개수의 최근 영상 구간
3. 현재 생성 중인 영상 구간
윈도가 이동하면 오래된 영상 구간은 캐시에서 제거됩니다. 따라서 영상이 10초이든 10분이든 한 번의 생성 단계에서 사용하는 문맥 크기는 거의 같습니다.

전체 과거 사용:
시간 증가 → 메모리 증가 → 계산량 증가

Sliding window:
시간 증가 → 윈도 크기는 고정 → 계산량 일정

Persistent reference context와 RoPE Repositioning

Persistent reference context
Sliding window에서 오래된 프레임을 계속 제거하면 최초 캐릭터 정보도 사라질 수 있습니다. 이를 막기 위해 다음 정보는 제거하지 않고 계속 유지합니다.
- 사용자가 제공한 최초 참조 이미지
- 최초로 생성한 video-audio block

이 정보가 캐릭터의 얼굴, 헤어스타일, 의상 등 정체성의 기준점 역할을 합니다. S2에서는 이를 sink block이라고 부릅니다.

RoPE Repositioning
윈도가 이동하면 캐시된 과거 구간의 상대적 위치도 바뀝니다. S1은 캐시를 처음부터 다시 계산하지 않고, 기존 key-value feature에 현재 윈도 기준의 RoPE 위치만 다시 적용합니다.
예를 들어 이전 단계에서 두 번째 위치였던 Chunk 8이 윈도 이동 후 첫 번째 위치가 되면, 위치 임베딩만 새 위치에 맞춰 갱신합니다.

RoPE: Rotary Position Embedding
Transformer가 각 토큰의 순서와 상대적 거리를 알도록 만드는 위치 인코딩 방법
어느 프레임이 먼저이고 서로 얼마나 떨어져 있는지는 알 수 없음 -> 각 토큰의 Query와 Key 벡터를 위치에 따라 서로 다른 각도로 회전시켜 위치 정보를 추가 - 토큰의 위치가 \(p\)일 때 Query와 Key 벡터 일부를 \(p\)에 비례하는 각도로 회전




TwinCache

Noisy cache
중간 denoising 단계에서 추출한, 약간의 노이즈가 남아 있는 표현입니다.
역할:
- 큰 움직임과 시간적 흐름 전달
- 작은 고주파 오류가 다음 구간으로 계속 복사되는 현상 억제
- 장시간 생성에서 흔들림과 드리프트 감소
  
Clean cache
최종 denoising이 끝난 깨끗한 표현입니다.
역할:
- 얼굴과 머리 모양 유지
- 의상과 질감 복원
- 캐릭터 정체성과 세부 묘사 유지
중간 denoising에서는 noisy cache를 사용하고, 마지막 단계에서는 clean cache로 교체

  
3-stage training: Bidirectional → Causal → DMD+PCM
학습 과정

① Bidirectional Teacher: 영상 전체를 보고 노이즈를 제거합니다. 미래 문맥까지 볼 수 있기 때문에 영상 품질은 높지만 실시간 생성에는 사용할 수 없음
과거 ← 현재 → 미래
고품질 생성 능력 학습
          ↓
② Causal Teacher: Bidirectional Teacher를 초기값으로 사용한 뒤, 각 구간이 과거만 볼 수 있도록 causal attention mask를 적용
과거 → 현재 → 미래
실시간 스트리밍 구조로 변환
          ↓
③ DMD + PCM
많은 denoising step → 3-step
실시간 속도 달성


DMD: Distribution Matching Distillation
교사 모델이 생성하는 고품질 영상의 분포를 학생 모델이 적은 단계로 따라 하도록 학습

PCM: DMD 학습을 안정화하는 규제 역할
DMD가 “결과의 전체적인 분포가 실제 영상처럼 보이는가?”를 맞춘다면, PCM의 perceptual loss는 “서로 인접한 생성 단계에서 영상의 의미와 시각적 구조가 안정적으로 유지되는가?”를 확인


Speech diarization

Raw Videos
→ Prefiltering
→ Clipping
→ Subject Filtering
→ Other Filtering
→ Diarization
→ Caption + Embedding

Prefiltering: 저해상도, 낮은 FPS, 음성·영상 불완전, 싱크가 맞지 않는 영상 제거
Clipping: 장면 전환이 없는 3~60초짜리 단일 숏으로 분할
Subject Filtering: 화면에 한 사람만 적절한 크기로 등장하는 영상 선택
Other Filtering: 자막·워터마크·흔들림·저화질·유해 콘텐츠 제거
Diarization: 지금 들리는 음성이 화면 속 인물의 목소리인지 판별하고, 화면 밖 음성이나 여러 사람의 중첩 발화 제거
Caption + Embedding: 남은 영상에 인물·행동·표정·대사·음향 설명을 붙여 학습 조건으로 변환

# Vidu S2-Avatar

2.1 Data Preparation
학습 데이터를 어떻게 만드는가
        ↓
2.2 Method
모델을 어떻게 학습하는가
        ↓
2.3 Inference Infrastructure
학습된 모델을 어떻게 실시간으로 실행하는가
        ↓
2.4 Agentic System
사용자의 지시와 참조 이미지를 어떻게 관리하는가

다양한 원본 영상


다양한 원본 영상
- Talking heads
- Film & TV
- Solo dance
- 2D animation
- 3D animation
        ↓
Clipping
        ↓
Filtering
        ↓
Speech Processing
        ↓
Temporal Dense Captioning
        ↓
Embedding
        ↓
S2-Avatar Training Data


S1의 데이터 파이프라인을 유지하면서 다음 세 가지를 강화했습니다.
1. 미세한 편집점 검출
2. 실제 선명도가 높은 영상 선별
3. 카메라가 움직이는 영상의 배경 안정화


① Data Collection
사용한 영상
- 라이브 방송·Talking-head 영상
- 영화·TV 영상
- 고품질 솔로 댄스 영상
- 2D 애니메이션
- 3D 애니메이션
목적
S1은 말하는 얼굴과 비교적 작은 몸동작 중심이었습니다. S2는 춤과 같은 큰 신체 움직임과 애니메이션 캐릭터까지 생성하기 위해 데이터 범위를 확장했습니다.
Talking head
→ 표정, 입 모양, 작은 제스처
Solo dance
→ 팔·다리·몸통의 큰 움직임
2D/3D animation
→ 비실사 캐릭터의 외형과 동작

② Video Clipping
목적
하나의 학습 클립 안에 장면 전환이 들어가지 않도록 원본 영상을 single-shot clip으로 자릅니다.
장면 전환이 포함되면 모델이 이를 실제 움직임으로 잘못 학습할 수 있습니다.
정상적인 움직임:
사람이 상자를 연다
→ 뚜껑을 올린다
→ 내용물을 꺼낸다

편집된 영상:
닫힌 상자
→ 갑자기 열린 상자
두 번째 경우의 중간 과정 생략을 검출하지 못하면 모델이 물체가 순간적으로 바뀌는 현상을 학습할 수 있습니다.
S2의 개선
일반적인 shot detector로 후보 편집점을 먼저 찾고, 후보 지점 주변 프레임을 VLM에 입력해 실제 장면 전환인지 다시 판단합니다.
원본 영상
   ↓
Shot detector
편집점 후보 탐색
   ↓
후보 주변 프레임 추출
   ↓
VLM의 2차 판정
   ↓
실제 편집점만 사용
민감도를 무조건 높이면 정상적인 움직임도 장면 전환으로 오판할 수 있습니다. VLM의 2차 검사를 통해 논문은 false rejection rate를 2% 이하로 유지한다고 설명합니다.


③ Video Filtering
S1의 여섯 가지 필터링 기준을 유지합니다.
필터	제거하는 영상
Subject detection	사람이 없거나 여러 인물이 섞인 영상
Frame cleanliness	자막, 워터마크, 광고 등이 포함된 영상
Visual quality	흐림, 깜빡임, 심한 압축 손상이 있는 영상
Content safety	부적절하거나 위험한 콘텐츠
Shot stability	카메라 움직임이 지나치게 복잡한 영상
Interactivity	인물의 의미 있는 행동이 부족한 영상


S2에서는 여기에 두 가지가 추가됩니다.
- High-Clarity Video Selection
- Background Stabilization


④ High-Clarity Video Selection
문제
파일 해상도가 높다고 실제 영상이 선명한 것은 아닙니다.
1080p + 심한 압축
→ 질감 손실, 블록 노이즈, 흐린 경계

720p + 약한 압축
→ 실제로 더 선명할 수 있음
따라서 S2는 해상도만 보고 학습 영상을 선택하지 않습니다.
평가 항목
1차 기술 조건:
- 해상도
- 프레임 레이트
추가 기술 조건:
- Codec
- Pixel format
- Bit depth
- Bitrate
영상 품질 조건:
- Texture detail
- Edge sharpness
- Compression artifact
- Technical quality
이 평가 결과를 가중합하여 품질 점수를 만들고, 기준값 이상인 영상만 720p 학습 데이터로 사용합니다.


⑤ Background Stabilization
문제
무한 스트리밍 생성에서는 배경이 시간에 따라 변형되는 background drift가 발생하기 쉽습니다. 따라서 S1은 고정 카메라 영상을 선호했습니다.
하지만 댄스 영상은 다음과 같은 카메라 움직임을 자주 포함합니다.
- Orbiting
- Dolly
- Zoom
- Subject tracking
이런 영상을 모두 제거하면 큰 몸동작을 학습할 데이터가 부족해집니다.
처리 과정
카메라가 부드럽게 움직이는 댄스 영상
        ↓
전경 인물 검출 및 마스킹
        ↓
배경 영역에서 특징점 추출
        ↓
프레임 사이의 카메라 변환 추정
        ↓
기하학적 변환으로 배경 정렬
        ↓
잘못 노출된 경계 부분 Crop
        ↓
인물이 화면 안에 있는지 재검사
인물을 마스킹하는 이유는 인물의 움직임을 카메라 움직임으로 잘못 계산하지 않기 위해서입니다.
카메라 이동이 크고 복잡해 parallax가 심한 영상은 제거하고, 회전·줌처럼 기하학적 변환으로 근사할 수 있는 영상만 안정화합니다.


⑥ Temporal Dense Captioning
S1 방식의 한계
S1은 정보를 항목별로 나누어 설명했습니다.
Subject: 파란 셔츠를 입은 여성
Environment: 회색 배경
Camera: 고정된 중간 클로즈업
Action: 손을 흔듦
장면 구성요소는 알 수 있지만 다음 정보가 약합니다.
- 행동이 언제 시작되는가
- 어떤 순서로 발생하는가
- 앞 행동이 뒤 행동에 어떤 영향을 주는가
- 지시가 바뀌는 시점은 언제인가
S2 방식
S2는 시간 순서가 포함된 dense caption을 사용합니다.
0~2.8초:
여성이 오른손을 얼굴 옆으로 들어
카메라를 향해 흔든다.

2.8~5초:
손을 내리고 미소를 지은 뒤
가볍게 고개를 끄덕인다.
각 캡션에는 다음 정보가 들어갑니다.
- 시간 범위
- 행동
- 행동의 결과
- 장면과 카메라 상태
- 앞뒤 사건의 관계
Event Boundary
S1의 chunk-level caption은 고정된 시간이나 음성 구간을 기준으로 나눴습니다. S2는 행동이 바뀌는 event boundary를 기준으로 자릅니다.
고정 시간 분할:
0~2초 / 2~4초 / 4~6초
→ 하나의 행동이 중간에서 잘릴 수 있음

Event boundary 분할:
손 올리기 / 손 흔들기 / 손 내리기 / 미소 짓기
→ 행동 단위로 조건 제공
이 방식은 새로운 지시에 더 빠르게 반응하고, 동작 전환을 자연스럽게 만드는 데 사용됩니다.
Multi-Agent Annotation
캡션은 단일 VLM이 한 번에 만들지 않습니다.
Object Detection + Speech Recognition
사실 정보 추출
        ↓
Recognition Agent
영상 속 대상과 행동 인식
        ↓
Description Agent
시간 순서에 맞게 문장 생성
        ↓
Verification Agent
내용이 영상과 일치하는지 검증
        ↓
Filtering Agent
환각이나 낮은 품질의 캡션 제거
발표 문장
S2는 장면 요소를 나열하는 캡션 대신 사건의 시작과 종료, 행동 순서와 결과를 포함하는 temporal dense caption을 사용합니다. 또한 고정 길이가 아니라 행동이 바뀌는 event boundary를 기준으로 영상을 구분합니다.


Method

## 전체 학습 과정

```text
① Audio-Visual Joint DiT
영상과 오디오를 공동 생성하는 기본 구조
        ↓
② Bidirectional I2V + R2V Training
고품질 생성 및 참조 이미지 조건 학습
        ↓
③ Causal Adaptation
Teacher Forcing + Diffusion Forcing
        ↓
④ Self-Replay Forcing
실제 자기회귀 오류 궤적을 재학습
        ↓
⑤ Preference Optimization
사람이 선호하는 품질과 동작으로 정렬
        ↓
⑥ One-step Super-Resolution Refiner
저해상도 latent를 720p용 고해상도로 변환
```


---

## ① Model Overview: Audio-Visual Joint DiT

Vidu S2-Avatar는 영상과 오디오를 함께 생성하는 Diffusion Transformer입니다.

기호:

- \(r\): 캐릭터의 참조 이미지
- \(x_i\): \(i\)번째 video-audio segment
- \(c_i\): 해당 구간의 캡션이나 사용자 지시
- \(x_t^{1:N}\): diffusion timestep \(t\)에서 노이즈가 포함된 video-audio latent
- \(\hat{x}_0^{1:N}\): 모델이 예측한 깨끗한 video-audio latent

모델:

\[
\hat{x}^{1:N}_0
=
f^{avatar}_{\theta}
(x^{1:N}_t,t,r,c^{1:N})
\]

의미:

```text
노이즈가 포함된 영상·오디오 latent
+ diffusion timestep
+ 참조 이미지
+ 구간별 지시
        ↓
Vidu S2-Avatar
        ↓
깨끗한 영상·오디오 latent
```

참조 이미지 \(r\)은 모든 시간 구간에 공통으로 제공됩니다. 이를 통해 영상이 길어져도 캐릭터의 얼굴과 외형을 유지합니다.

---

## ② Bidirectional I2V + R2V Training

하나의 모델을 두 작업으로 동시에 학습합니다.

### I2V: Image-to-Video

참조 이미지 \(r\)이 목표 영상의 첫 번째 프레임입니다.

```text
영상의 첫 프레임
        ↓
같은 장면과 인물이 이어지는 영상 생성
```

목적:

- 첫 프레임에서 자연스럽게 이어지는 움직임
- 장면과 카메라의 연속성
- 최초 이미지와 출력 영상의 연결

### R2V: Reference-to-Video

참조 이미지 \(r\)이 목표 영상과 정확히 같은 프레임일 필요는 없습니다. 캐릭터의 정체성과 외형을 알려주는 별도의 이미지입니다.

```text
캐릭터 참조 이미지
        ↓
다른 포즈와 장면에서도 같은 캐릭터 생성
```

목적:

- 얼굴 정체성 유지
- 머리 모양과 의상 유지
- 다양한 포즈와 움직임 생성

### Segment-wise Conditional Supervision

일반적인 bidirectional model은 전체 영상에 하나의 프롬프트를 제공합니다.

```text
전체 영상:
“여성이 춤을 춘다”
```

S2는 각 segment에 별도의 조건 \(c_i\)를 제공합니다.

```text
Segment 1: 오른손을 올린다
Segment 2: 양손을 흔든다
Segment 3: 몸을 회전한다
Segment 4: 원래 자세로 돌아온다
```

이를 통해 각 시점에서 수행해야 할 행동이 더 명확해지고 instruction following이 향상됩니다.

---

## ③ Hybrid Teacher Forcing and Diffusion Forcing

Bidirectional model은 미래 구간까지 볼 수 있으므로 실시간 스트리밍에 사용할 수 없습니다. 따라서 temporal attention을 block-wise causal attention으로 변경합니다.

```text
Bidirectional:
과거 ← 현재 → 미래

Causal:
과거 → 현재 → 미래
```

\(i\)번째 segment는 다음 정보만 봅니다.

- 참조 이미지 \(r\)
- 현재 지시 \(c_i\)
- 현재 noisy segment \(x^i_{t_i}\)
- 유효한 과거 상태 \(x^{<i}_{\tau_i}\)

\[
\hat{x}^{i}_0
=
f^{causal}_{\theta}
(x^i_{t_i},t_i,r,c_i,x^{<i}_{\tau_i},\tau_i)
\]

### Teacher Forcing

과거 문맥으로 깨끗한 정답 영상을 사용합니다.

```text
깨끗한 정답 과거
        ↓
현재 segment 생성
```

장점:

- 학습이 안정적
- 동작과 음성 동기화를 배우기 쉬움

단점:

- 실제 추론에서는 정답 과거가 존재하지 않음

### Diffusion Forcing

과거 문맥에 서로 다른 수준의 노이즈를 넣습니다.

```text
노이즈가 포함된 과거
        ↓
현재 segment 생성
```

장점:

- 불완전한 과거 상태에 견디는 능력 학습
- 자기회귀 생성 중 오류 누적에 더 강함

학습 중 두 방식 중 하나를 정해진 확률로 선택합니다.

### 한계

이 단계만으로는 모델이 실제 추론에서 직접 생성한 과거 분포를 완전히 경험하지 못합니다. 이를 해결하는 것이 Self-Replay Forcing입니다.

---

## ④ Self-Replay Forcing

### 해결하려는 문제

실제 추론에서는 모델이 자신의 출력을 다시 입력으로 사용합니다.

```text
모델이 만든 Chunk 1
        ↓
Chunk 1을 보고 Chunk 2 생성
        ↓
Chunk 1, 2를 보고 Chunk 3 생성
```

작은 오류가 반복해서 다음 구간에 전달되면 장시간 생성에서 다음 문제가 생깁니다.

- 얼굴 변화
- 의상 변화
- 배경 drift
- 손과 신체 구조 붕괴
- 동작 반복
- 영상 전체 collapse

Teacher Forcing과 단순 Diffusion Forcing은 모델이 실제로 생성한 장기 궤적을 그대로 사용하지 않기 때문에 train-test gap이 남습니다.

### SRF 전체 과정

```text
① 현재 Student가 긴 영상을 자기회귀 방식으로 생성
        ↓
② 생성된 block과 KV cache를 계산 그래프에서 detach
        ↓
③ 생성된 영상 구간에 다시 노이즈 추가
        ↓
④ 여러 구간을 하나의 causal pass로 replay
        ↓
⑤ Replay된 모든 구간에 DMD supervision 적용
        ↓
⑥ 뒤쪽 구간의 손실이 앞쪽 replay 구간까지 역전파
```

### 1단계: Autoregressive Rollout

현재 student가 실제 inference 방식으로 긴 영상을 생성합니다.

```text
Student → Chunk 1 → Chunk 2 → Chunk 3 → Chunk 4
```

이를 **on-policy trajectory**라고 합니다. 현재 모델이 실제로 만들어내는 분포이기 때문입니다.

### 2단계: Detach

원래 rollout의 block과 KV cache를 계산 그래프에서 분리합니다.

이유:

- 긴 rollout 전체의 계산 그래프를 저장하면 메모리 사용량이 너무 큼
- 원래 생성 과정을 모두 역전파하면 학습 비용이 지나치게 커짐

### 3단계: Re-noising

모델이 생성한 각 block에 Diffusion Forcing 방식으로 다시 노이즈를 추가합니다.

```text
생성된 Chunk 1 → noisy Chunk 1
생성된 Chunk 2 → noisy Chunk 2
생성된 Chunk 3 → noisy Chunk 3
```

### 4단계: Gradient-enabled Replay

다시 노이즈화된 여러 block을 하나의 causal pass에서 재생합니다.

원래 rollout은 detach되어 있지만 replay 내부의 segment들은 같은 계산 그래프에 연결됩니다.

```text
Replay Chunk 1
      ↓ gradient
Replay Chunk 2
      ↓ gradient
Replay Chunk 3
```

따라서 Chunk 3에서 발생한 오류가 Chunk 1과 Chunk 2의 replay 표현까지 영향을 줄 수 있습니다.

### DMD + Perceptual Loss

SRF의 손실:

\[
L_{\mathrm{SRF}}
=
L_{\mathrm{DMD}}
+
L_{\mathrm{perc}}
\]

- \(L_{\mathrm{DMD}}\): student의 생성 분포를 teacher·데이터 분포에 맞춤
- \(L_{\mathrm{perc}}\): mode collapse를 억제하고 시각적 다양성과 구조를 보존

### 기존 Self-Forcing과의 차이

| Self-Forcing | Self-Replay Forcing |
|---|---|
| 모델이 생성한 과거를 조건으로 사용 | 모델이 생성한 전체 궤적을 다시 학습 |
| 과거가 clean 상태 | 각 segment를 다시 noising |
| 과거가 계산 그래프에서 분리 | replay segment끼리는 계산 그래프로 연결 |
| segment 사이 gradient가 차단됨 | replay 내부에서 cross-block gradient 전달 |

### 함께 설명할 구조

SRF에서도 S1의 다음 구조를 유지합니다.

- Sink block
- Sliding-window context
- Noisy KV cache



---

## ⑤ Preference Optimization

객관적인 diffusion loss만으로는 사람이 느끼는 자연스러움과 지시 이행 능력을 충분히 최적화하기 어렵습니다. 따라서 두 단계에서 preference optimization을 수행합니다.

### Bidirectional Stage: Diffusion DPO

Bidirectional model에 diffusion-based DPO를 적용합니다.

개선 대상:

- Visual fidelity
- Facial expressiveness
- Motion naturalness
- Audio-visual synchronization

목적:

```text
더 좋은 Bidirectional Teacher
        ↓
더 좋은 Causal Student
```

### Streaming Stage: Streaming NFT

Causal backbone에는 Streaming Negative-aware Fine-Tuning을 적용합니다.

학습 과정:

```text
자기회귀 rollout 생성
        ↓
rollout detach
        ↓
일부 trajectory replay
        ↓
보상 기반 최적화
```

개선 대상:

- 영상 품질
- 동작 제어
- 지시 이행
- 실제 streaming 상태에서의 안정성

핵심은 보상 학습도 정답 영상 상태가 아니라 모델이 실제로 만든 자기회귀 trajectory에서 수행한다는 것입니다.

### 발표 문장

> Bidirectional 단계에서는 DPO로 기본 생성 품질과 표현력을 높이고, streaming 단계에서는 모델이 직접 생성한 trajectory에 Streaming NFT를 적용해 실제 추론 환경에서의 동작 제어와 지시 이행을 개선합니다.

---

## ⑥ One-Step Super-Resolution Refiner

### 필요한 이유

720p latent를 backbone이 처음부터 직접 생성하면 계산량이 커져 실시간 속도를 유지하기 어렵습니다.

따라서 역할을 분리합니다.

```text
Causal Backbone
저해상도 latent에서
움직임과 시간 구조 생성
        ↓
One-step Refiner
고해상도 latent로 변환하고
세부 묘사 복원
        ↓
720p 영상
```

### Refiner 입력 생성

Backbone의 저해상도 출력 \(\hat{x}^{i}_{0,LR}\)을 latent 공간에서 upsampling한 뒤, 고정된 refinement timestep \(t_r\)에 해당하는 노이즈를 추가합니다.

\[
z^i_{t_r}
=
(1-t_r)U(\hat{x}^{i}_{0,LR})
+
t_r\epsilon^i
\]

- \(U\): latent-space upsampling
- \(\epsilon^i\): Gaussian noise
- \(t_r\): 고정된 refinement timestep

Refiner는 이 noisy high-resolution latent를 한 번에 정제합니다.

### Asymmetric Cache

Backbone과 Refiner가 서로 다른 noise level의 과거 cache를 사용합니다.

| 모듈 | 사용하는 캐시 | 역할 |
|---|---|---|
| Causal Backbone | High-noise cache | 큰 움직임과 장기 시간 구조 전달 |
| Refiner | Low-noise high-resolution cache | 얼굴, 의상, 질감, 정체성 복원 |

\[
\tau_B > \tau_R
\]

- \(\tau_B\): backbone cache의 noise level
- \(\tau_R\): Refiner cache의 noise level

### High-noise cache

고주파 세부 오류에 덜 민감하므로 작은 시각적 오류가 다음 구간으로 계속 복사되는 것을 줄입니다. 장기 움직임과 시간적 구조를 안정적으로 전달합니다.

### Low-noise high-resolution cache

깨끗한 외형 정보를 많이 포함하므로 다음 정보를 보존합니다.

- 얼굴 특징
- 머리 모양
- 의상 질감
- 캐릭터 정체성
- 고해상도 세부 정보

### 발표 문장

> Backbone은 high-noise cache를 사용해 장기 움직임을 안정적으로 전달하고, Refiner는 low-noise high-resolution cache를 사용해 얼굴과 의상의 세부 정보를 복원합니다. 이를 통해 시간적 안정성과 720p 화질을 서로 분리해 처리합니다.


---

# 2.3 Inference Infrastructure

학습 방법만으로는 실시간 속도를 달성할 수 없기 때문에 연산 자체를 최적화합니다.

```text
Efficient Attention
        +
W8A8 Linear Layers
        +
Kernel Fusion
        +
CUDA Graphs
        +
Multi-GPU Parallelism
        ↓
실시간 저지연 추론
```

---

## ① Efficient Attention

Attention은 Diffusion Transformer의 주요 연산 병목입니다.

S2는 모든 layer에 같은 근사 attention을 적용하지 않고 layer별로 선택합니다.

- SageAttention
- SpargeAttention
- Sparse-Linear Attention

```text
오차에 민감한 layer
→ 정확도가 높은 방식

오차에 덜 민감한 layer
→ 더 공격적인 양자화·희소화 방식
```

목적:

- 민감한 layer의 품질 유지
- 덜 민감한 layer의 계산량 감소
- 전체 attention latency 감소

### 발표 문장

> 모든 layer를 동일하게 근사하면 품질이 크게 떨어질 수 있으므로, S2는 layer 민감도에 따라 서로 다른 attention 가속 방식을 적용합니다.

---

## ② Per-block W8A8 GEMM

Linear layer의 weight와 activation을 8-bit로 계산합니다.

- W8: 8-bit weight
- A8: 8-bit activation
- GEMM: 행렬 곱셈

### 일반적인 양자화의 문제

Per-tensor나 per-channel 방식에서는 소수의 매우 큰 outlier가 전체 양자화 범위를 결정할 수 있습니다.

```text
대부분의 값: -1 ~ 1
Outlier: 20

전체 범위가 -20 ~ 20으로 설정
→ 대부분의 작은 값을 정밀하게 표현하지 못함
```

### S2의 해결

값을 작은 block으로 나누고 block마다 scaling factor를 사용합니다.

```text
Tensor
→ Block 1: 자체 scale
→ Block 2: 자체 scale
→ Block 3: 자체 scale
```

outlier의 영향이 해당 block으로 제한되므로 속도를 높이면서 영상 품질을 유지할 수 있습니다.

### 발표 문장

> S2는 weight와 activation을 8-bit로 계산하되, 전체 tensor가 아니라 작은 block마다 scale을 적용하여 outlier 때문에 정밀도가 떨어지는 문제를 줄입니다.

---

## ③ Kernel Fusion

GPU에서 작은 연산을 각각 실행하면 매번 kernel launch와 메모리 접근이 발생합니다.

기존 방식:

```text
RMSNorm
→ 결과를 GPU memory에 저장
→ Elementwise operation 실행
→ 다시 저장
```

Kernel fusion:

```text
RMSNorm + Elementwise operation
→ 하나의 Triton/CUDA kernel에서 실행
```

효과:

- Kernel launch 횟수 감소
- 중간 tensor의 global memory 저장 감소
- Synchronization overhead 감소
- Memory bandwidth 사용 감소

---

## ④ CUDA Graphs

Streaming inference는 동일한 연산 순서를 반복합니다.

```text
Backbone step
→ Attention
→ Linear
→ Norm
→ 다음 denoising step
```

CUDA Graph는 이 실행 순서를 처음 한 번 기록하고 이후 그대로 재생합니다.

```text
기존:
CPU가 GPU kernel을 하나씩 호출

CUDA Graph:
여러 kernel 호출을 하나의 graph launch로 실행
```

Kernel 실행 시간이 짧을수록 launch overhead의 비중이 커지므로 반복 구조를 가진 streaming 모델에서 효과적입니다.

---

## ⑤ Multi-GPU Parallelism

Ulysses-style context parallelism을 사용합니다.

긴 video-audio token sequence를 여러 GPU에 나눕니다.

```text
전체 sequence
        ↓
GPU 1: token 구간 A
GPU 2: token 구간 B
GPU 3: token 구간 C
GPU 4: token 구간 D
```

Attention 전후에는 all-to-all communication을 통해 sequence-parallel layout과 head-parallel layout 사이에서 activation을 재배치합니다.

통신 tensor도 양자화하여 다음을 줄입니다.

- GPU 간 전송량
- 통신 latency
- 전체 inference latency

### 발표 문장

> S2는 긴 video-audio token sequence를 여러 GPU에 분산하고, GPU 사이에서 교환하는 activation도 양자화하여 병렬화로 인한 통신 비용을 줄입니다.

---

# 2.4 Agentic System

함께 볼 이미지: **Figure 3**

전체 구조:

```text
사용자 텍스트
사용자 음성
초기 이미지
추가 참조 이미지
        ↓
VLM Agent
        ↓ Prompt
Vidu S2-Avatar
        ↓ Video Frames
VLM Agent가 결과 검사
        ↓ Feedback
다음 Prompt 수정
```

음성 지시는 먼저 텍스트로 변환됩니다. VLM agent는 입력을 해석해 생성 프롬프트를 작성하고, 생성 결과를 다시 검사하여 다음 프롬프트를 조정합니다.

---

## ① Prompt Generation

프롬프트를 다음 요소로 분리해 관리합니다.

- 캐릭터의 정체성과 외형
- 표정
- 시선
- 자세
- 행동
- 계속 들고 있어야 하는 물체
- 유지해야 하는 배경과 의상

사용자가 변경하지 않은 정보는 이전 상태를 유지합니다.

### 예시

사용자 지시:

```text
“컵을 집은 다음 웃어.”
```

프롬프트 변화:

```text
1단계:
오른손을 컵 쪽으로 뻗어 컵을 집는다.

2단계:
오른손에 컵을 계속 든 상태로 미소 짓는다.
```

두 번째 프롬프트에서 “컵을 계속 들고 있다”를 명시하지 않으면 모델이 웃는 동작으로 전환할 때 컵을 없애거나 내려놓을 수 있습니다.

### 발표 문장

> Agent는 새로운 지시에 필요한 속성만 변경하고 나머지 상태를 프롬프트에 계속 유지합니다. 이를 통해 여러 단계의 행동에서도 물체, 의상 및 캐릭터 상태가 사라지는 것을 방지합니다.

---

## ② Visual Feedback

VLM agent는 생성된 프레임을 시간 순서대로 검사합니다.

판정:

- 행동 완료
- 일부 수행
- 지시와 다른 결과
- 판단 불확실

```text
행동 완료
→ 완료된 상태를 다음 prompt에 기록
→ 같은 행동 반복 방지

일부 수행
→ 남은 행동을 다시 지시

잘못 수행
→ prompt 수정 후 재지시

불확실
→ 완료로 처리하지 않음
```

여러 프레임에서 반복적으로 같은 결과가 관찰되는지와 confidence도 확인합니다.

### 예시

```text
지시: 컵을 집어라

한 프레임에서 손이 컵 근처에 있음
→ 아직 완료로 판단하지 않음

여러 프레임에서 컵을 손에 들고 있음
→ 완료로 판단
→ 다음 prompt: 컵을 든 상태 유지
```

---

## ③ Reference Image 처리

Agent는 참조 이미지의 종류를 먼저 분류합니다.

- Handheld object
- Background
- Clothing

그다음 초기 이미지와 참조 이미지를 비교해 무엇을 변경하고 무엇을 유지할지 프롬프트로 작성합니다.

### Object Reference

함께 볼 이미지: **Figure 4(a)**

```text
초기 캐릭터 이미지
+ 파란 컵 참조 이미지
        ↓
파란 컵 생성 또는 기존 컵 교체
```

캐릭터 외형과 배경은 유지하고 컵만 변경합니다.

### Scene Reference

함께 볼 이미지: **Figure 4(b)**

단순히 한 프레임에서 배경을 즉시 교체하는 것이 아니라 장면 전환 행동을 프롬프트로 작성할 수 있습니다.

```text
캐릭터가 기존 장면을 떠남
        ↓
참조 이미지의 새로운 장면으로 이동
```

### Clothing Reference

참조 의상의 색상, 형태, 재질을 분석하고 캐릭터가 해당 옷을 입도록 프롬프트를 구성합니다. 얼굴이나 변경 요청이 없는 요소는 유지합니다.

---

## ④ 액세서리 상태 관리

함께 볼 이미지: **Figure 4(c)**

예시:

```text
모자를 벗어라
→ 손을 모자로 이동
→ 모자를 들어 올림
→ 손에 모자를 들고 있음

다시 모자를 써라
→ 손에 든 모자를 머리로 이동
→ 모자를 착용
→ 이후 prompt에서도 착용 상태 유지
```

Agent는 행동뿐 아니라 행동이 끝난 후의 상태까지 프롬프트에 기록합니다.

```text
행동:
모자를 벗는다

결과 상태:
모자를 손에 들고 있다
```

이 구분이 없으면 모델이 행동 직후 물체나 액세서리를 사라지게 만들 수 있습니다.

---



# 3 Vidu S2-Editing

전체 구조:

```text
3.1 Data Preparation
편집 전·후 영상 쌍을 어떻게 만드는가
        ↓
3.2 Method
원본 움직임을 유지하면서 원하는 부분만 편집하는 모델
        ↓
3.3 Inference Infrastructure
입력 영상과 편집 결과의 지연을 줄이는 GPU 스케줄링
```

Vidu S2-Editing이 수행하는 작업은 네 가지입니다.

- Style Transfer | 영상 전체의 화풍을 변경
- Subject Replacement | 영상 속 인물을 다른 인물·캐릭터로 교체
- Background Replacement | 인물은 유지하고 배경만 교체
- Virtual Try-On | 인물이 입은 옷을 참조 의상으로 교체

---

# 3.1 Data Preparation

## 전체 데이터 구성

함께 볼 이미지: **Figure 2 오른쪽**

```text
S2-Avatar 데이터 파이프라인을 통과한 영상
        ↓
더 엄격한 품질 필터링
        ↓
Label Balancing
        ↓
총 800,000개 영상
        ↓
4개 작업에 200,000개씩 배분
```

Label Balancing | 특정 인물, 동작, 장면, 의상 등에 데이터가 편중되지 않도록 학습 샘플의 분포를 조절하는 것

네 작업의 데이터는 서로 겹치지 않습니다.

```text
Style Transfer           200K
Subject Replacement      200K
Background Replacement   200K
Virtual Try-On           200K
                         ────
                         800K
```

Disjoint Subsets | 같은 영상이 여러 작업용 데이터에 중복으로 포함되지 않도록 완전히 분리한 데이터 집합

### 발표 문장

> S2-Editing은 Avatar 파이프라인에서 선별한 고품질 영상에 더 엄격한 필터링과 label balancing을 적용합니다. 이후 80만 개 영상을 네 편집 작업에 서로 겹치지 않도록 20만 개씩 배분합니다.

---

## ① Style Transfer 데이터 생성

### 문제

영상 편집 모델을 학습하려면 다음과 같은 정답 쌍이 필요합니다.

```text
Source Video
원본 실사 영상
        +
Reference Image
목표 화풍
        ↓
Target Video
원본 움직임을 유지한 스타일 영상
```

하지만 같은 움직임을 가진 실사 영상과 스타일 영상의 정확한 쌍은 현실에서 구하기 어렵습니다.

따라서 별도의 V2V 모델을 먼저 학습하여 편집 데이터를 합성합니다.

V2V | Video-to-Video, 입력 영상의 구조나 움직임을 조건으로 새로운 영상을 생성하는 모델

---

## Stage 1: Reconstruction Model 학습

함께 볼 이미지: **Figure 2 오른쪽 Stage 1**

```text
Raw Video
   ├──────────────→ Target Video
   │
   ├→ 한 프레임 샘플링 → Reference Image
   │
   └→ NormalCrafter → Surface-Normal Video
                              │
Reference Image ──────────────┤
                              ↓
                          V2V Model
                              ↓
                     Reconstructed Video
                              ↓
                   원본 영상과 Reconstruction Loss
```

### Surface-Normal Video

Surface Normal | 물체 표면이 어느 방향을 향하는지를 RGB 색상 등으로 표현한 기하학적 정보

일반적인 RGB 영상에는 다음 정보가 섞여 있습니다.

- 인물과 물체의 형태
- 색상
- 재질
- 조명
- 그림자
- 영상 스타일

Surface-normal video에는 색상과 화풍보다 다음 정보가 중심적으로 남습니다.

- 얼굴과 신체의 3차원 형태
- 물체 표면 방향
- 포즈
- 움직임
- 공간 구조

NormalCrafter | 일반 RGB 영상으로부터 시간적으로 일관된 surface-normal video를 추정하는 모델

### 학습 입력

- \(v_{\text{normal}}\): 원본 영상에서 추출한 surface-normal video
- \(i_{\text{ref}}\): 원본 영상에서 샘플링한 한 프레임

### 학습 목표

V2V 모델이 normal video의 구조와 움직임을 따르면서 reference image의 외형을 적용해 원본 영상을 복원하도록 학습합니다.

```text
Normal Video
→ 구조와 움직임 제공

Reference Image
→ 색상, 질감, 외형 제공

결과
→ 원본 RGB 영상 복원
```

### 왜 이렇게 학습하는가?

모델이 다음 두 정보를 분리해서 사용하도록 만들기 위해서입니다.

```text
영상의 움직임·구조
← Surface-Normal Video

영상의 외형·스타일
← Reference Image
```

### 발표 문장

> 먼저 원본 영상을 normal video와 reference image로 분해한 뒤, 두 조건으로 원본 영상을 복원하도록 V2V 모델을 학습합니다. 이를 통해 normal video는 움직임과 구조를, reference image는 외형과 스타일을 담당하도록 만듭니다.

---

## Stage 2: Style Transfer Pair 생성

함께 볼 이미지: **Figure 2 오른쪽 Stage 2**

Stage 1에서 학습한 V2V 모델에 원본 reference image 대신 stylized reference image를 넣습니다.

```text
원본 영상의 Normal Video
→ 원래 포즈와 움직임 유지

Stylized Reference Image
→ 새로운 색상, 질감, 화풍 제공

        ↓
학습된 V2V Model
        ↓
Stylized Video
```

예시:

```text
원본 동작:
컵을 들고 움직이는 여성

Stylized Reference:
수채화 스타일의 여성 이미지

결과:
원래 동작과 카메라 움직임은 유지하면서
영상 전체가 수채화 스타일로 변경
```

생성된 데이터 쌍:

```text
Source: 원본 RGB 영상
Target: 동일한 움직임의 스타일 영상
Reference: 목표 스타일 이미지
```

이 데이터를 이용해 최종 S2-Editing 모델을 학습합니다.

논문은 다음을 포함해 50개 이상의 스타일에 일반화한다고 설명합니다.

- Cel shading
- Cyberpunk
- Monet-style painting
- Sketch
- Chinese gongbi painting

Gongbi | 세밀한 선과 정교한 채색을 사용하는 중국 전통 회화 기법

### 발표 문장

> 학습된 V2V 모델에 원본 normal video와 새로운 스타일 이미지를 입력하면, 원본의 구조와 움직임은 유지하면서 외형만 바뀐 영상을 만들 수 있습니다. 이 합성 결과를 원본과 짝지어 style-transfer 학습 데이터로 사용합니다.

---

## ② 나머지 편집 작업의 데이터 생성

다음 작업도 비슷한 방식으로 편집 전·후 영상 쌍을 만듭니다.

- Subject Replacement
- Background Replacement
- Virtual Try-On

각 작업에 맞는 video editing model을 fine-tuning하여 후보 결과를 생성합니다.

Fine-tuning | 이미 학습된 모델을 특정 작업이나 데이터에 맞게 추가 학습하는 것

추가로 여러 공개 편집 모델을 사용합니다.

- Bernini
- SCAIL-2
- Wan2.1 VACE
- SAMA-14B
- CoinVE-Edit

### 여러 모델을 사용하는 이유

모델마다 잘하는 작업과 샘플이 다르기 때문입니다.

```text
모델 A
→ 의상 교체는 잘하지만 배경 교체가 약함

모델 B
→ 배경 교체는 잘하지만 인물 정체성 유지가 약함

모델 C
→ 특정 애니메이션 스타일에서 강함
```

한 모델의 출력만 사용하지 않고 여러 모델이 만든 후보를 비교합니다.

```text
동일한 Source + Reference
        ↓
여러 편집 모델로 후보 생성
        ↓
Post-filtering
        ↓
Comparative Selection
        ↓
가장 품질이 좋은 편집 결과 선택
```

Post-filtering | 생성 결과에서 심한 왜곡, 편집 실패, 시간적 불일치 등이 있는 샘플을 제거하는 과정

Comparative Selection | 여러 모델이 생성한 후보를 서로 비교하여 가장 좋은 결과를 선택하는 과정

### 최종 학습 데이터

```text
Source Video
+ Reference Image
+ Editing Instruction
+ Selected Edited Video
```

### 발표 문장

> 나머지 세 작업은 여러 편집 모델을 이용해 후보 영상을 생성하고, 후처리 필터링과 모델 간 비교를 통해 가장 품질이 높은 source-edited video pair만 학습 데이터로 사용합니다.

---

## ③ Data Captioning

각 캡션에는 두 종류의 정보가 들어갑니다.

1. 어떤 편집 작업인지
2. 참조 이미지에서 어떤 속성을 적용해야 하는지

### Style Transfer

```text
Task:
Style transfer

Reference Attributes:
수채화 질감, 낮은 채도,
부드러운 붓 터치
```

### Virtual Try-On

```text
Task:
Virtual try-on

Reference Attributes:
긴소매 흰색 셔츠,
꽃무늬 장식,
면 소재
```

### Subject Replacement

```text
Task:
Subject replacement

Reference Attributes:
여성 캐릭터의 얼굴,
검은 머리,
주황색 기모노
```

### Background Replacement

```text
Task:
Background replacement

Reference Attributes:
파리의 야외 카페,
빨간 차양,
석조 건물
```

캡션에는 보존 조건도 명시합니다.

```text
변경할 부분:
의상

보존할 부분:
얼굴, 신체 움직임, 포즈,
카메라 움직임, 배경
```

Preservation Constraint | 사용자가 편집을 요청하지 않은 영상 요소는 원본 그대로 유지하라는 조건

---

## Source Video를 VLM에 보여주지 않는 이유

캡션을 만들 때 VLM에는 source video를 입력하지 않습니다.

VLM | Vision-Language Model, 이미지나 영상을 보고 내용을 텍스트로 설명하거나 추론하는 모델

VLM이 source video를 보면 원본의 구체적인 내용을 캡션에 적을 수 있습니다.

예:

```text
Source Video:
파란 셔츠를 입은 여성이 걷는 영상

잘못된 Caption:
파란 셔츠를 입은 여성이 걷고 있으며,
이를 흰색 셔츠로 변경한다.
```

이 경우 텍스트 조건에 원본의 “파란 셔츠” 정보가 들어갑니다. 모델은 원본 영상뿐 아니라 텍스트에서도 파란 셔츠 정보를 받아 편집 목표와 충돌할 수 있습니다.

이를 Text Conditioning Leakage라고 볼 수 있습니다.

Text Conditioning Leakage | 원본 영상의 내용이 텍스트 조건에 포함되어, 모델이 편집 대상의 기존 속성을 텍스트를 통해 다시 전달받는 문제

따라서 캡션 생성 시 다음 정보만 사용합니다.

```text
편집 작업 종류
+ Reference Image
+ Preservation Constraint
```

Source video의 내용은 텍스트로 설명하지 않습니다.

### 발표 문장

> 편집 캡션은 작업 종류와 참조 이미지의 속성, 그리고 보존 조건만 설명합니다. Source video는 VLM에 보여주지 않아 원본의 편집 대상 정보가 텍스트 조건으로 다시 유입되는 것을 방지합니다.

---

# 3.2 Method

## 전체 모델 구조

```text
Source Video
→ 원래 포즈, 움직임, 카메라 정보

Reference Image
→ 적용할 인물, 옷, 배경, 스타일

Editing Instruction
→ 무엇을 바꾸고 무엇을 유지할지 지정

Noisy Target Video
→ 편집 결과가 생성될 초기 noisy latent

        ↓
Vidu S2-Editing DiT
        ↓
Clean Edited Video Latent
```

---

## ① Model Overview

Vidu S2-Editing은 Diffusion Transformer를 사용합니다.

DiT | Diffusion Transformer, 노이즈가 포함된 이미지·영상 latent를 Transformer로 반복 정제하여 깨끗한 결과를 생성하는 모델

기호:

- \(s^{1:N}\): source video 조건
- \(r\): 선택적 reference image
- \(c_i\): \(i\)번째 segment의 편집 지시
- \(x_t^{1:N}\): 노이즈가 포함된 target video latent
- \(\hat{x}_0^{1:N}\): 모델이 예측한 깨끗한 편집 영상 latent

수식:

\[
\hat{x}^{1:N}_0
=
f^{edit}_{\theta}
(x^{1:N}_t,t,s^{1:N},r,c^{1:N})
\]

입력별 역할:

| 입력 | 역할 |
|---|---|
| Source video \(s\) | 포즈, 움직임, 카메라, 공간 구조 |
| Reference image \(r\) | 새 스타일, 인물, 의상, 배경의 외형 |
| Instruction \(c\) | 편집 대상과 보존 대상을 지정 |
| Noisy target \(x_t\) | 정제되어 최종 편집 영상이 될 latent |
| Timestep \(t\) | 현재 denoising 단계 표시 |

---

## Conditioning Token 구성

Source video와 reference image를 각각 token으로 인코딩합니다.

Conditioning Token | 생성 결과를 제어하기 위해 모델에 함께 입력하는 조건 정보의 token 표현

```text
Noisy Target Tokens
+ Source Video Tokens
+ Reference Image Tokens
+ Instruction Tokens
        ↓
Transformer
```

서로 다른 token stream은 의미와 좌표 체계가 다릅니다.

- Target token: 생성할 영상의 위치
- Source token: 원본 영상의 위치와 시간
- Reference token: 단일 참조 이미지의 위치
- Instruction token: 언어 의미

따라서 각각에 condition-specific RoPE를 적용합니다.

Condition-Specific RoPE | source, target, reference처럼 서로 다른 조건 token에 각각 적합한 공간·시간 위치 정보를 부여하는 방식

### 발표 문장

> Source video, reference image, target video는 서로 다른 시간·공간 구조를 가지므로 각각 별도의 RoPE 체계로 위치를 표현한 뒤 하나의 Transformer 입력으로 결합합니다.

---

## ② Bidirectional Video Editing Training

먼저 전체 target sequence를 양방향으로 볼 수 있는 편집 모델을 학습합니다.

Bidirectional Training | 현재 프레임이 과거와 미래의 target frame을 모두 참조할 수 있는 학습 방식

```text
Target 1 ↔ Target 2 ↔ Target 3 ↔ Target 4
```

이 단계에서는 전체 편집 영상의 외형과 움직임을 일관되게 만드는 강한 prior를 학습합니다.

Prior | 모델이 학습 데이터로부터 미리 습득한 영상의 외형, 움직임 및 시간적 일관성에 대한 지식

---

## Frame-Aligned Attention

Vidu S2-Editing의 핵심 구조입니다.

### 기본 원리

각 target frame은 동일한 시점의 source frame만 직접 참조합니다.

```text
Source Frame 1 ─→ Target Frame 1
Source Frame 2 ─→ Target Frame 2
Source Frame 3 ─→ Target Frame 3
Source Frame 4 ─→ Target Frame 4
```

다음 연결은 제한됩니다.

```text
Source Frame 1 ─X→ Target Frame 3
Source Frame 4 ─X→ Target Frame 2
```

Frame-Aligned Attention | 편집 결과의 각 프레임이 시간적으로 동일한 source frame과만 정보를 교환하도록 제한한 attention

### 왜 필요한가?

Source video에는 원본의 정확한 움직임과 타이밍이 들어 있습니다.

시간이 맞지 않는 source frame을 unrestricted attention으로 참조하면 다음 문제가 생길 수 있습니다.

- 서로 다른 포즈가 섞임
- 움직임의 시점이 밀림
- 잔상과 ghosting 발생
- 손이나 팔이 중복됨
- 원본 동작이 제대로 보존되지 않음

Frame-aligned attention은 target frame이 동일 시점의 source frame에서 다음 정보를 가져오게 합니다.

- 포즈
- 신체 위치
- 카메라 위치
- 물체 배치
- 움직임 타이밍

### Target끼리의 관계

Target video token끼리는 양방향으로 상호작용할 수 있습니다.

```text
Target 1 ↔ Target 2 ↔ Target 3 ↔ Target 4
```

따라서 편집된 외형이 영상 전체에서 일관되게 유지됩니다.

### Reference Image와의 관계

모든 target frame이 reference-image token을 볼 수 있습니다.

```text
                    ┌→ Target Frame 1
Reference Image ────┼→ Target Frame 2
                    ├→ Target Frame 3
                    └→ Target Frame 4
```

이를 통해 참조 의상, 인물, 배경, 화풍을 전체 영상에 일관되게 적용합니다.

### 전체 구조

```text
Source Frame i
    │
    │ 같은 시간의 움직임·구조
    ↓
Target Frame i ← Reference Image
                    │
                    └ 외형·스타일
```

### 발표 문장

> 각 target frame은 같은 시점의 source frame에서 포즈와 움직임을 가져오고, 모든 target frame은 공통 reference image에서 새로운 외형을 가져옵니다. 이를 통해 움직임은 원본에 맞추고 외형만 참조 이미지에 맞게 변경합니다.

### 넣으면 좋은 그림

논문에 전용 구조 그림이 없으므로 다음과 같이 직접 그리면 됩니다.

```text
Source Video        Edited Target Video
   S1 ─────────────────→ T1
   S2 ─────────────────→ T2
   S3 ─────────────────→ T3
   S4 ─────────────────→ T4

Reference Image
   R ─────→ T1, T2, T3, T4
```

Source-target 연결은 가로 실선, reference-target 연결은 다른 색 점선으로 표시하면 됩니다.

---

## ③ Causal Streaming Training

Bidirectional model은 미래 target frame을 볼 수 있으므로 실시간 편집에는 사용할 수 없습니다.

따라서 temporal attention을 block-wise causal attention으로 변경합니다.

Block-Wise Causal Attention | 현재 영상 구간이 현재 및 과거 구간만 보고 미래 구간은 볼 수 없도록 제한한 attention

```text
Bidirectional:
Target 1 ↔ Target 2 ↔ Target 3 ↔ Target 4

Causal:
Target 1 → Target 2 → Target 3 → Target 4
```

현재 segment는 다음 조건을 사용합니다.

- 현재까지 들어온 source video
- reference image
- 현재 editing instruction
- 현재까지 생성된 편집 영상 history

```text
Source Segment i
+ Reference Image
+ Editing Prompt i
+ Edited History < i
        ↓
Edited Segment i 생성
```

---

## Hybrid Forcing

Avatar 모델과 동일하게 Teacher Forcing과 Diffusion Forcing을 섞어 학습합니다.

### Teacher Forcing

Teacher Forcing | 깨끗한 정답 과거 편집 영상을 history로 제공하는 학습 방식

```text
Ground-Truth Edited Segment 1, 2
                ↓
Edited Segment 3 학습
```

장점:

- 안정적인 학습
- 편집 결과와 움직임을 쉽게 정렬

단점:

- 실제 inference에서는 정답 history가 없음

### Diffusion Forcing

Diffusion Forcing | 과거 영상 segment에 서로 다른 수준의 노이즈를 넣어 불완전한 history에 견디도록 학습하는 방식

```text
Noisy Edited Segment 1, 2
             ↓
Edited Segment 3 학습
```

목적:

- 이전 편집 영상에 오류가 있어도 다음 구간이 무너지지 않게 함
- 장시간 편집에서 오류 누적 억제

---

## Self-Replay Forcing

편집 모델도 Avatar와 동일한 SRF를 사용합니다.

Self-Replay Forcing | 모델이 실제로 생성한 자기회귀 trajectory를 다시 노이즈화하고 미분 가능한 형태로 재생하여 학습하는 방법

```text
① 현재 Editing Student가 긴 편집 영상 생성
        ↓
② Rollout과 KV cache를 detach
        ↓
③ 생성된 편집 영상에 다시 noise 추가
        ↓
④ 여러 segment를 causal하게 replay
        ↓
⑤ DMD supervision 적용
```

Autoregressive Rollout | 모델이 자신의 이전 출력을 다음 입력으로 사용하면서 긴 영상을 순차적으로 생성하는 과정

Detach | 해당 계산 결과는 사용하지만 원래 생성 과정으로 gradient를 역전파하지 않도록 계산 그래프에서 분리하는 것

Re-noising | 모델이 생성한 깨끗한 결과에 diffusion noise를 다시 추가하는 것

Differentiable Replay | 다시 노이즈화한 영상 구간을 계산 그래프에 연결한 상태로 재생하여 gradient가 흐르게 하는 과정

DMD Supervision | student의 생성 분포가 teacher 또는 실제 데이터의 분포에 가까워지도록 하는 학습 신호

### 편집 모델에서 SRF가 필요한 이유

편집 스트림에서도 오류가 누적됩니다.

```text
초기 옷 색상 오류
        ↓
다음 segment가 잘못된 옷을 history로 사용
        ↓
색상과 형태가 계속 변화
        ↓
원래 의상 또는 다른 의상으로 drift
```

SRF는 모델이 실제로 만든 이런 오류 궤적을 다시 학습합니다.

개선 대상:

- 편집된 인물 정체성 유지
- 의상 형태 유지
- 배경 일관성
- 시간적 안정성
- 편집 지시 이행

### 발표 문장

> Causal adaptation으로 실시간 편집 구조를 만든 뒤, Self-Replay Forcing으로 모델이 실제 편집 과정에서 생성한 오류 궤적을 다시 학습합니다. 이를 통해 장시간 스트리밍에서도 편집 결과가 원래 모습으로 돌아가거나 시간에 따라 변형되는 현상을 줄입니다.


# 4 Real-Time Spatial Video Generation and Editing

전체 구조:

```text
4.1 Spatial Video Generation
단안 Avatar 영상을 좌안·우안 영상으로 변환
        ↓
4.2 Spatial Video Editing
일반 영상 또는 기존 공간 영상을 실시간으로 편집
        ↓
4.3 Discussion
현재 한계와 향후 panoramic spatial video
```

---

# Spatial Video란?

Spatial Video | 왼쪽 눈과 오른쪽 눈에 약간 다른 영상을 보여줘 깊이감을 만드는 영상

```text
Left-eye View  → 왼쪽 눈
Right-eye View → 오른쪽 눈
        ↓
두 영상의 시차를 뇌가 해석
        ↓
깊이, 거리, 물체 크기 인식
```

일반 영상은 두 눈에 같은 이미지를 보여주지만, 공간 영상은 좌·우 시점에 차이를 줍니다. 이를 통해 캐릭터가 화면 안에 있는 것이 아니라 사용자 주변 공간에 존재하는 것처럼 느끼게 합니다.

---

# 4.1 Real-Time Spatial Video Generation

## 목적

Vidu S2-Avatar는 기본적으로 단안 영상을 생성합니다.

Monocular Video | 하나의 카메라 시점으로 구성된 일반적인 2D 영상

단안 영상만으로는 깊이감이 약하기 때문에, 생성된 스트림을 실시간으로 좌안·우안 영상으로 변환합니다.

---

## Generation Pipeline

```text
텍스트·음성·참조 이미지·영상 history
        ↓
Vidu S2-Avatar
        ↓
다음 Monocular Video Chunk 생성
        ↓
프레임별 Depth Estimation
        ↓
Depth를 Horizontal Disparity로 변환
        ↓
반대 방향으로 Image Warping
        ↓
Left View + Right View
        ↓
시간 동기화 후 VR로 전송
```

### ① Monocular Video 생성

Vidu S2-Avatar가 다음 조건을 이용해 일반 단안 영상 구간을 생성합니다.

- 현재 텍스트 지시
- 사용자 음성
- 참조 이미지
- 이전 영상 history
- 기타 제어 조건

---

### ② Depth Estimation

Depth Estimation | 각 픽셀이 카메라에서 얼마나 가까이 또는 멀리 있는지 추정하는 과정

```text
인물 얼굴 → 가까운 깊이
인물 몸    → 중간 깊이
배경       → 먼 깊이
```

각 프레임에서 depth map을 생성합니다.

Depth Map | 픽셀별 거리를 밝기나 수치로 표현한 지도

---

### ③ Horizontal Disparity 변환

Disparity | 동일한 물체가 좌안과 우안 영상에서 서로 다른 수평 위치에 나타나는 차이

깊이값을 좌우 시차로 변환합니다.

```text
가까운 물체 → 좌우 위치 차이가 큼
먼 물체     → 좌우 위치 차이가 작음
```

원래 단안 프레임을 중앙 시점으로 간주하고, 픽셀을 반대 방향으로 이동시켜 두 영상을 만듭니다.

```text
Center View
   ├→ 왼쪽 방향 Warping → Left View
   └→ 오른쪽 방향 Warping → Right View
```

Image Warping | 깊이 또는 기하학 정보에 따라 원본 이미지의 픽셀 위치를 이동시키는 처리

---

### ④ Warping Artifact 보정

픽셀을 이동하면 원래 보이지 않던 영역이 드러나면서 빈 공간이 생깁니다.

```text
원본:
인물 뒤의 배경이 가려져 있음

Warping:
인물이 옆으로 이동
→ 뒤에 있던 배경 정보가 없음
→ 빈 영역 발생
```

Disocclusion | 시점이 바뀌면서 이전에는 물체에 가려졌던 영역이 새롭게 드러나는 현상

S2는 저지연 처리를 위해 가벼운 보정으로 다음 문제를 줄입니다.

- 물체 경계의 찢어짐
- 깊이 경계 주변 artifact
- 새롭게 드러난 영역의 hole
- 좌우 영상의 불일치

---

### ⑤ Temporal Depth Stabilization

프레임마다 depth를 독립적으로 추정하면 깊이값이 흔들릴 수 있습니다.

```text
Frame 1: 얼굴 깊이 0.35
Frame 2: 얼굴 깊이 0.42
Frame 3: 얼굴 깊이 0.33
```

실제 얼굴은 거의 움직이지 않아도 VR에서는 앞뒤로 떨리는 것처럼 보입니다.

Temporal Stabilization | 연속 프레임 사이에서 depth 변화가 급격하게 흔들리지 않도록 보정하는 과정

이를 통해 좌우 시차와 체감 깊이를 시간적으로 안정시킵니다.

### 발표 문장

> Vidu S2는 Avatar가 생성한 단안 프레임에서 깊이를 추정하고, 깊이에 따라 픽셀을 좌우로 이동시켜 좌안·우안 영상을 만듭니다. 이후 warping으로 생긴 빈 영역과 깊이 흔들림을 보정해 실시간 공간 영상으로 출력합니다.

---

# 4.2 Real-Time Spatial Video Editing

Vidu S2-Editing은 두 종류의 입력을 지원합니다.

```text
① Monocular Input
일반 2D 영상

② Stereoscopic Input
이미 좌안·우안 영상이 존재하는 공간 영상
```

---

## ① Monocular Input

```text
일반 Monocular Stream
        ↓
Vidu S2-Editing
스타일·의상·인물·배경 편집
        ↓
편집된 Monocular Stream
        ↓
Depth Estimation + Warping
        ↓
Left View + Right View
```

핵심은 **먼저 편집하고 나중에 공간 영상으로 변환**한다는 것입니다.

예:

```text
일반 카메라 영상
        ↓
의상을 참조 이미지의 옷으로 변경
        ↓
편집된 영상의 depth 추정
        ↓
좌안·우안 영상 생성
```

---

## ② Stereoscopic Input

Stereoscopic Video | 이미 좌안 영상과 우안 영상으로 구성된 공간 영상

기존 공간 영상을 편집할 때는 좌안과 우안을 가로로 연결합니다.

```text
Left View + Right View
        ↓
Horizontal Concatenation
        ↓
하나의 Wide Video
        ↓
Vidu S2-Editing으로 한 번에 편집
        ↓
Wide Video를 다시 분리
        ↓
Edited Left View + Edited Right View
```

Horizontal Concatenation | 두 영상을 좌우 방향으로 이어 붙여 하나의 넓은 영상으로 만드는 것

두 시점을 따로 편집하지 않고 한 번에 처리하는 이유는 동일한 참조 이미지와 편집 조건을 적용해 좌우 결과의 일관성을 높이기 위해서입니다.

```text
좌안을 따로 편집
우안을 따로 편집
→ 인물·의상·배경이 서로 다르게 생성될 위험

좌우 영상을 함께 편집
→ 같은 외형과 스타일을 두 시점에 동시에 적용
```

### 지원 편집

두 입력 방식 모두 다음 작업을 지원합니다.

- Style Transfer
- Virtual Try-On
- Character Replacement
- Background Replacement

영상 파일뿐 아니라 카메라 또는 VR headset passthrough 영상에도 적용할 수 있습니다.

Passthrough | 헤드셋 외부 카메라로 촬영한 현실 영상을 사용자 디스플레이에 실시간으로 보여주는 기능

### 발표 문장

> 단안 입력은 먼저 편집한 뒤 공간 영상으로 변환합니다. 기존 공간 영상은 좌안과 우안을 하나의 넓은 영상으로 결합해 한 번에 편집하고 다시 분리함으로써 두 시점의 외형과 스타일을 일관되게 유지합니다.

---

# 4.3 Discussion

## 현재 한계

### ① 더 높은 해상도 필요

VR에서는 영상이 사용자의 시야 대부분을 차지합니다. 일반 모니터보다 픽셀이 더 크게 보이므로 낮은 해상도에서는 흐림과 픽셀 구조가 쉽게 드러납니다.

따라서 공간 영상은 일반 단안 영상보다 더 높은 해상도가 필요합니다.

### ② 더 낮은 지연 시간 필요

특히 passthrough 환경에서 사용자가 머리를 움직였는데 화면이 늦게 따라오면 실제 움직임과 영상이 어긋납니다.

```text
실제 머리 움직임
        ↓ 지연
화면 시점 갱신
        ↓
멀미·불편함 발생
```

따라서 높은 영상 품질과 낮은 end-to-end latency를 동시에 달성하는 것이 핵심 과제입니다.

---

## 미래 방향: Panoramic Spatial Video

현재 방식은 기본적으로 고정된 카메라 시점의 좌안·우안 영상을 생성합니다.

Fixed-View Spatial Video | 사용자가 정해진 방향에서만 깊이감을 느낄 수 있는 공간 영상

논문이 제안하는 향후 방향은 panoramic spatial video입니다.

Panoramic Spatial Video | 사용자가 머리를 돌려 여러 방향을 자유롭게 볼 수 있는 공간 영상

```text
현재:
정면의 좌안·우안 영상 생성

미래:
사용자가 고개를 돌림
        ↓
새로운 방향의 장면을 실시간 생성
        ↓
360° 공간 탐색
```

이 기능이 발전하면 다음 요소를 실시간 생성하는 메타버스 형태로 확장할 수 있습니다.

- 장면
- 캐릭터
- 사건
- 사용자 행동에 따른 환경 변화

기존의 3D 모델링, 재질 제작, 애니메이션, 장면 구축 일부를 실시간 영상 생성으로 대체하는 것이 장기적인 목표입니다.

---

# 4장 핵심 구조

```text
[Vidu S2-Avatar]
Monocular Video 생성
        ↓
Depth Estimation
        ↓
Horizontal Disparity
        ↓
Left/Right Warping
        ↓
Hole Filling + Depth Stabilization
        ↓
Real-Time Spatial Video
```

```text
[Vidu S2-Editing]

Monocular Input:
편집 → 공간 영상 변환

Stereoscopic Input:
좌우 결합 → 한 번에 편집 → 좌우 분리
```

# 5 Experiments

실험은 크게 두 모델을 평가합니다.

- **Vidu S2-Avatar** | 실시간 디지털 캐릭터 생성
- **Vidu S2-Editing** | 실시간 영상 편집

---

## 5.1 평가 방법

### Avatar

**StreamAV-Bench**를 사용해 다음을 평가합니다.

- 영상·오디오 품질
- 음성과 입 모양의 동기화
- 사용자 지시 이행
- 장시간 인물·배경 일관성

### Editing

네 개의 벤치마크를 사용합니다.

- **Sparkle-Bench** | 지시 이행과 원본 움직임 보존
- **OpenVE-Bench** | 스타일·배경 편집
- **RefVIE-Bench** | 참조 이미지를 이용한 인물·배경 교체
- **ViViD** | Virtual Try-On

### 인간 평가

**GSB | 두 모델의 결과를 사람이 직접 비교하는 평가**

- Good | Vidu S2가 더 좋음
- Same | 차이가 없음
- Bad | 비교 모델이 더 좋음

---

# 5.2 공개 벤치마크 결과

## Vidu S2-Avatar

함께 볼 이미지: **Table 1**

Vidu S2-Avatar는 StreamAV-Bench의 모든 보고 지표에서 1위를 기록합니다.

주요 결과:

- 영상 품질 1위
- 오디오 품질 1위
- 음성·입 모양 동기화 1위
- 사용자 지시 이행 1위
- 인물·배경 일관성 1위

특히 다음 수치가 중요합니다.

```text
Subject Consistency    0.998
Background Consistency 0.993
```

즉, 영상이 길어져도 인물과 배경이 거의 변하지 않았다는 결과입니다.

---

## Vidu S2-Editing

함께 볼 이미지: **Table 2~4**

주요 결과:

```text
Sparkle Overall     3.74  → 1위
OpenVE Overall      4.42  → 1위
RefVIE Overall      3.78  → 1위
Joint Overall       4.26  → 1위
Virtual Try-On      9.95  → 1위
```

Virtual Try-On의 **VFIDI는 낮을수록 좋습니다.**

```text
Vidu S2-Editing  9.95
CatV2TON        19.51
ViViD           21.80
```

결론:

> Vidu S2-Editing은 실시간 streaming 모델이지만 일부 offline 편집 모델보다도 높은 성능을 기록했습니다.

---

# 5.3 인간 선호도 평가

## Avatar

함께 볼 이미지: **Figure 5, Figure 6**

Figure 5에서는 영상을 10초부터 90초까지 생성하면서 품질을 평가합니다.

Vidu S2는 시간이 길어져도 다음 항목에서 가장 높은 점수를 유지합니다.

- 인물 일관성
- 영상 품질
- 움직임 품질
- 표정 표현
- 전체 품질

전체 선호도:

```text
vs Runway   85.7%
vs PixVerse 100%
vs HeyGen   100%
```

---

## Editing

함께 볼 이미지: **Figure 7**

전체 선호도:

```text
vs Decart-Lucy2.5  72.7%
vs XMax-X2.0       86.7%
```

시간적 일관성 점수:

```text
Vidu S2-Editing  3.56
Decart-Lucy2.5   2.59
XMax-X2.0        1.67
```

즉, Vidu S2는 한 프레임의 편집 품질뿐 아니라 영상 전체에서 편집 결과를 유지하는 능력이 우수합니다.

---

# 5.4 정성 비교

## Avatar

함께 볼 이미지: **Figure 8**

Vidu S2는 얼굴, 머리, 의상, 손 구조를 안정적으로 유지합니다.

비교 모델에서는 다음 문제가 발생합니다.

- 얼굴과 신체 비율 변화
- 헤어스타일·의상 변화
- 손가락·눈썹 왜곡

---

## Editing

| 이미지 | 평가 작업 | Vidu S2가 잘 유지한 요소 |
|---|---|---|
| Figure 9 | Style Transfer | 얼굴, 포즈, 배경 스타일 |
| Figure 10 | Virtual Try-On | 의상 무늬, 신체 움직임, 손과 옷의 관계 |
| Figure 11 | Subject Replacement | 참조 인물, 원본 포즈, 카메라 움직임 |
| Figure 12 | Background Replacement | 전경 인물, 행동, 참조 배경 구조 |
| Figure 13 | Spatial Video | 좌안·우안 영상의 깊이감 |

---

# 핵심 결론

```text
Vidu S2-Avatar
→ 품질, 동기화, 지시 이행, 장시간 일관성 모두 1위

Vidu S2-Editing
→ 네 가지 편집 벤치마크 모두 1위
→ 실시간 모델이지만 offline 모델보다도 높은 성능

인간 평가
→ Avatar와 Editing 모두 기존 상용 모델보다 높은 선호도
```


6 Conclusion
We present Vidu S2, which comprises Vidu S2-Avatar, a real-time interactive digital-character model, and
Vidu S2-Editing, a real-time video editing model, and we further explore the feasibility of real-time spatial video generation. Compared with Vidu S1, Vidu S2-Avatar supports real-time 720p video generation,
dynamic references that can be updated at any moment, and stronger instruction following, such as dancing. Vidu S2-Editing edits a video stream in real time, covering style transfer, virtual try-on, character
replacement, and background replacement. Experiments show that Vidu S2 outperforms all baselines.